# Plant Disease Classification

Training notebook for a custom CNN and an EfficientNetB0 transfer-learning model on plant leaf images.

**Sections:**
1. Imports & Config
2. Data Loading
3. Exploratory Data Analysis
4. Custom CNN Model
5. EfficientNetB0 Transfer Learning
6. Evaluation
7. Save Models

## 1. Imports & Config

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = "../data"
MODELS_DIR = "../models"
ASSETS_DIR = "../assets"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ASSETS_DIR, exist_ok=True)

## 2. Data Loading

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "train"), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)
val_gen = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "val"), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)
test_gen = test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "test"), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False
)

class_names = list(train_gen.class_indices.keys())
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")

## 3. Exploratory Data Analysis

In [ ]:
images, labels = next(train_gen)

plt.figure(figsize=(12, 8))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Custom CNN Model

In [ ]:
def build_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

cnn_model = build_cnn((*IMG_SIZE, 3), num_classes)
cnn_model.summary()

In [ ]:
cnn_history = cnn_model.fit(train_gen, validation_data=val_gen, epochs=10)

## 5. EfficientNetB0 Transfer Learning

In [ ]:
def build_efficientnet(input_shape, num_classes):
    base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=input_shape)
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

effnet_model = build_efficientnet((*IMG_SIZE, 3), num_classes)
effnet_model.summary()

In [ ]:
effnet_history = effnet_model.fit(train_gen, validation_data=val_gen, epochs=10)

## 6. Evaluation

In [ ]:
def evaluate_model(model, test_gen, class_names, save_path=None):
    preds = model.predict(test_gen)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes

    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

evaluate_model(effnet_model, test_gen, class_names, save_path=os.path.join(ASSETS_DIR, "confusion_matrix.png"))

## 7. Save Models

In [ ]:
cnn_model.save(os.path.join(MODELS_DIR, "plant_disease_cnn.keras"))
effnet_model.save(os.path.join(MODELS_DIR, "plant_disease_efficientnetb0.keras"))
print("Models saved.")